In [1]:
import sys
from pathlib import Path

# Notebook ka folder
NOTEBOOK_DIR = Path().resolve()

# Project root = parent folder
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Add project root to import path
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

import warnings as w
w.filterwarnings("ignore")

Project root: D:\Langchain_LangGraph_03-12-2025\MyProject


In [2]:
"""
Production-Grade RAG System with:
1. Hybrid Search (Semantic + Keyword)
2. Reranking (Cohere)
3. LangGraph Integration
4. Evaluation Metrics
5. Complete Pipeline

Run in Jupyter Notebook
"""


'\nProduction-Grade RAG System with:\n1. Hybrid Search (Semantic + Keyword)\n2. Reranking (Cohere)\n3. LangGraph Integration\n4. Evaluation Metrics\n5. Complete Pipeline\n\nRun in Jupyter Notebook\n'

#### LangChain related Library

In [3]:
#now importing all the Module which is used to build the AI Model.
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from exception import CustomException
from logger_config import logger
import os,sys

#using openai chat model and embedding models
from langchain_openai import ChatOpenAI,OpenAIEmbeddings

#using groq chat model 
from langchain_groq import ChatGroq

#using open source chat model from hugging Face
from langchain_huggingface import ChatHuggingFace,HuggingFaceEmbeddings,HuggingFaceEndpoint

from config import *

from langchain_core.runnables import RunnableBranch,RunnableLambda,RunnableParallel,RunnableSequence,RunnablePassthrough

[2025-12-18 13:10:26,772]-config_variable.py-INFO -Loading the environment Variable
[2025-12-18 13:10:26,775]-config_variable.py-INFO -Environment Variable successfully Loaded


In [4]:
%pwd

'd:\\Langchain_LangGraph_03-12-2025\\MyProject\\notebooks'

#### LanGraph related Library

In [5]:
#import Langgraph related Modules
import langgraph
from langgraph.graph import StateGraph,START,END
from dataclasses import dataclass
from typing import TypedDict
from typing import Literal,List,Annotated
from langchain_core.messages import AnyMessage,AIMessage,HumanMessage,ToolMessage

from pydantic import BaseModel #using this class we can perform validation to schema

from langgraph.prebuilt import tool_node,tools_condition #in this class we put all tools together
#tools_condition wrt to tool msg it will route the flow data to ttol node to perform execution

from langchain_core.tools import tool,Tool,StructuredTool

from langgraph.graph.message import BaseMessage #this is special class which hold every mesaage init.

import time

## step:1) defining the models components

In [6]:
model1 = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.2 #we call as creative parameter
)
model1

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001E6416DD340>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E641868EB0>, root_client=<openai.OpenAI object at 0x000001E6416DD730>, root_async_client=<openai.AsyncOpenAI object at 0x000001E641868FA0>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [7]:
model2 = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2 #we call as creative parameter
)
model2

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001E641FE74C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E64200BF40>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
llm = HuggingFaceEndpoint(  
repo_id="meta-llama/Llama-3.1-8B-Instruct",  
task="text-generation",  
max_new_tokens=512,  
do_sample=False,  
repetition_penalty=1.03,  
)  

model3 = ChatHuggingFace(llm=llm, verbose=True)
model3

ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='meta-llama/Llama-3.1-8B-Instruct', repetition_penalty=1.03, stop_sequences=[], server_kwargs={}, model_kwargs={}, model='meta-llama/Llama-3.1-8B-Instruct', client=<InferenceClient(model='meta-llama/Llama-3.1-8B-Instruct', timeout=120)>, async_client=<InferenceClient(model='meta-llama/Llama-3.1-8B-Instruct', timeout=120)>, task='text-generation'), model_id='meta-llama/Llama-3.1-8B-Instruct', model_kwargs={})

In [9]:
### Hugging face Embedding Models.
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpointEmbeddings
hug_emb_model = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-large-en-v1.5",
    task = "feature-extraction",
)
hug_emb_model

HuggingFaceEndpointEmbeddings(client=<InferenceClient(model='BAAI/bge-large-en-v1.5', timeout=None)>, async_client=<InferenceClient(model='BAAI/bge-large-en-v1.5', timeout=None)>, model='BAAI/bge-large-en-v1.5', provider=None, repo_id='BAAI/bge-large-en-v1.5', task='feature-extraction', model_kwargs=None, huggingfacehub_api_token=None)

In [10]:
from langchain_openai import OpenAIEmbeddings
emb_model = OpenAIEmbeddings(
    model="text-embedding-3-small"  
)
emb_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001E6431F5D60>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001E6420D1EE0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [11]:
#calling the Vision model who will convert the image to text.
image_model = ChatOpenAI(
    model="gpt-4o"
)
image_model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001E643215400>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E643215520>, root_client=<openai.OpenAI object at 0x000001E6432154C0>, root_async_client=<openai.AsyncOpenAI object at 0x000001E6432153A0>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

### Loading PDF file parsing Pdf to get images table flowchart

In [3]:
from pathlib import Path
import pdfplumber
pdf_path = Path("D:/Langchain_LangGraph_03-12-2025/PDFFolder/attention.pdf")
pdf_path

WindowsPath('D:/Langchain_LangGraph_03-12-2025/PDFFolder/attention.pdf')

In [13]:
#now loading the pdf and extracting the text from each page as well as Images too.
with pdfplumber.open(pdf_path) as pdf:
    #how many pages present in pdf
    third_page = pdf.pages[2]
    content = third_page.extract_text(
        x_tolerance=2,  #x_tolerance → horizontal spacing fix
        y_tolerance=3,  #y_tolerance → line spacing fix
        layout=True     #page geometry respect karta hai
    )
    print(content)

### similarly extracting images from this pages

In [14]:
#now loading the pdf and extracting the Images too.
import fitz #PyMuPDF
doc = fitz.open(pdf_path)
#to see the total pages exist in PDF
print(f"Total Pages In PDF exist is: {len(doc)}")

thirdpage_index = 2  # 3rd page (2-based indexing)
page = doc[thirdpage_index]

images = page.get_images(full=True)
print(f"Images found on page 3: {len(images)}")

Total Pages In PDF exist is: 15
Images found on page 3: 1


In [15]:
'''
(xref, smask, width, height, bpc, colorspace, alt, name, filter, decode_parms)
example:-
| Index | Value         | Meaning                                                                    |
| ----- | ------------- | -------------------------------------------------------------------------- |
| `0`   | `128`         | **xref** → Image ka unique reference ID (PDF internal object number)       |
| `1`   | `175`         | **smask** → Soft mask (transparency reference). `0` ya number ho sakta hai |
| `2`   | `1520`        | **width** → Image width in pixels                                          |
| `3`   | `2239`        | **height** → Image height in pixels                                        |
| `4`   | `8`           | **bpc** → Bits per component (8-bit color depth)                           |
| `5`   | `DeviceRGB`   | **colorspace** → RGB color model                                           |
| `6`   | `''`          | **alt** → Alternate text (mostly empty in research PDFs)                   |
| `7`   | `Im1`         | **name** → Internal image name in PDF                                      |
| `8`   | `FlateDecode` | **filter** → Compression type (PNG-like)                                   |
| `9`   | `0`           | **decode_parms** → Decode parameters (usually unused)                      |

'''

"\n(xref, smask, width, height, bpc, colorspace, alt, name, filter, decode_parms)\nexample:-\n| Index | Value         | Meaning                                                                    |\n| ----- | ------------- | -------------------------------------------------------------------------- |\n| `0`   | `128`         | **xref** → Image ka unique reference ID (PDF internal object number)       |\n| `1`   | `175`         | **smask** → Soft mask (transparency reference). `0` ya number ho sakta hai |\n| `2`   | `1520`        | **width** → Image width in pixels                                          |\n| `3`   | `2239`        | **height** → Image height in pixels                                        |\n| `4`   | `8`           | **bpc** → Bits per component (8-bit color depth)                           |\n| `5`   | `DeviceRGB`   | **colorspace** → RGB color model                                           |\n| `6`   | `''`          | **alt** → Alternate text (mostly empty in resear

In [16]:
print(images)

[(128, 175, 1520, 2239, 8, 'DeviceRGB', '', 'Im1', 'FlateDecode', 0)]


In [17]:
#now extracting image and storing into dirctory.
output_dir = Path(r"D:\Langchain_LangGraph_03-12-2025\ImageDir")
for img_no, img in enumerate(images, start=1):
    xref = img[0]
    #wrt to image ref unique ID extracting image from pdf
    base_image = doc.extract_image(xref)
    
    # Extract raw image bytes from the PDF image object
    image_bytes = base_image["image"]

    # Get the image file extension (png, jpeg, etc.)
    image_ext = base_image["ext"]
    
    #loaction to save image
    img_file = output_dir / f"page3_img{img_no}.{image_ext}"
    
    with open(img_file, "wb") as f:
        f.write(image_bytes)

print("✅ Images from page 3 extracted successfully")
    
    

✅ Images from page 3 extracted successfully


In [18]:
import camelot

tables = camelot.read_pdf(
    str(pdf_path),
    pages="6",          # only page 6 or all means all tables from PDF
    flavor="stream"     # research papers ke liye best
)
print("Camelot tables found:", tables.n)

Camelot tables found: 1


In [22]:
table_dir = Path(r"D:\Langchain_LangGraph_03-12-2025\TableDir")
for i, table in enumerate(tables, start=1):
    df = table.df
    #converting dataframe object to markdown table.
    md_table = df.to_markdown(index=False)
    
    md_file = table_dir / f"camelot_page6_table_{i}.md"
    
    # Write the Markdown table content to disk using UTF-8 encoding
    md_file.write_text(md_table, encoding="utf-8")
    

#### below block i am texting can i extract table wrt pdfplumber its a hit and trial process

In [25]:
with pdfplumber.open(pdf_path) as pdf:
    page_index = 5  # 6th page (5-based)
    page = pdf.pages[page_index]

    tables = page.extract_tables()

print("pdfplumber tables found:", len(tables))

pdfplumber tables found: 0


In [ ]:
import pytesseract
from PIL import Image

# If Tesseract is not in PATH (Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Admin\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"

# Image path
image_path = Path(r"D:\Langchain_LangGraph_03-12-2025\ImageDir\page3_img1.png")

# Load image
image = Image.open(image_path)

# Extract text using OCR
text = pytesseract.image_to_string(image)

print(text)

Output
Probabilities

Add & Norm
Feed
Forward
Add & Norm

Multi- Head
Attention

Add & Norm

Add & Norm

Nx | Gada. Norm
Add & Norm Masked
Multi- Head Multi-Head
Attention Attention
SE a, of

Positional Positional

Encoding @ © © @ Encoding
Input Output

Embedding Embedding

Inputs Outputs
(shifted right)




: 